# Simulation of our mesh system.

Assumption that relay is all in the LoRa form, with a set IDEAL_RANGE value. There is a random set of entrances, based on the TRAIL_LENGTH and INTERNET_EVERY



No Relay

In [26]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TRAIL_LENGTH = 300  # miles
INTERNET_EVERY = 150  # miles
num_trail_entrances = 3
IDEAL_RANGE = 1  # miles

SIMULATION_TIME = 150  # days
HIKER_MEAN_SPEED = 2  # mph
HIKER_SPEED_SPREAD = 0.5  # mph; speed ~ Uniform(mean - spread, mean + spread)
MEAN_HIKING_TIME = 8  # hours
NUM_HIKERS_PER_DAY = 2500 // 120
NUM_TRIALS = 1000

DT = 1  # hours per simulation step

MESSAGE_DAY = 3


def jittered(a, b, n, rng):
    edges = np.linspace(a, b, n + 1)
    return edges[:-1] + rng.random(n) * (edges[1] - edges[0])


def run_trial(rng):
    """Returns (send_time, all_have_time, internet_has_message, counts, internet_step)."""
    entrances = jittered(0, TRAIL_LENGTH, num_trail_entrances - 2, rng)
    entrances = np.sort(np.concatenate([entrances, [0, TRAIL_LENGTH]]))
    n_entr = len(entrances)
    N = int(round(NUM_HIKERS_PER_DAY * SIMULATION_TIME))

    # ---- Hikers as arrays ----
    start_hour = rng.integers(0, SIMULATION_TIME, N) * 24.0
    start_idx = rng.integers(0, n_entr, N)

    direction = rng.choice([-1, 1], N)
    direction[start_idx == 0] = 1
    direction[start_idx == n_entr - 1] = -1

    low = np.where(direction == 1, start_idx + 1, 0)
    high = np.where(direction == 1, n_entr, start_idx)
    dest_idx = rng.integers(low, high)

    start_pos = entrances[start_idx]
    dest_pos = entrances[dest_idx]
    speed = rng.uniform(HIKER_MEAN_SPEED - HIKER_SPEED_SPREAD,
                        HIKER_MEAN_SPEED + HIKER_SPEED_SPREAD, N)
    stop_hour = start_hour + np.abs(dest_pos - start_pos) / speed

    msg = np.zeros(N, dtype=bool)
    on = np.zeros(N, dtype=bool)
    pos = start_pos.copy()
    internet_has_message = False
    send_time = None
    counts = []
    internet_step = None

    # ---- Simulation loop ----
    for t in np.arange(0, SIMULATION_TIME * 24, DT):
        was_on = on
        prev_pos = pos

        on = (start_hour <= t) & (t < stop_hour)
        pos = np.where(t < start_hour, start_pos,
                       np.where(on, start_pos + direction * speed * (t - start_hour),
                                dest_pos))

        # Reached or passed an entrance this step -> sync with internet
        lo = np.minimum(prev_pos, pos)
        hi = np.maximum(prev_pos, pos)
        hit = (was_on | on) & ((entrances >= lo[:, None]) &
                               (entrances <= hi[:, None])).any(axis=1)

        if not internet_has_message and (hit & msg).any():
            internet_has_message = True
            internet_step = len(counts)
        if internet_has_message:
            msg |= hit
            msg |= ~on  # anyone off the trail (finished or not started) is on the internet

        # A random on-trail hiker sends the message on MESSAGE_DAY
        if send_time is None and t >= MESSAGE_DAY * 24 and on.any():
            msg[rng.choice(np.flatnonzero(on))] = True
            send_time = t

        # Spread between on-trail hikers within IDEAL_RANGE of a carrier
        act = np.flatnonzero(on)
        carriers = np.sort(pos[act][msg[act]])
        if 0 < carriers.size < act.size:
            p = pos[act]
            k = np.searchsorted(carriers, p)
            left = carriers[np.maximum(k - 1, 0)]
            right = carriers[np.minimum(k, carriers.size - 1)]
            near = np.minimum(np.abs(p - left), np.abs(p - right)) <= IDEAL_RANGE
            msg[act[near]] = True

        # % of hikers currently on the trail who have the message
        n_on = on.sum()
        counts.append(100 * (msg & on).sum() / n_on if n_on else np.nan)

        # Stop once everyone has it
        if send_time is not None and msg.all():
            return send_time, t, internet_has_message, np.array(counts), internet_step

    return send_time, None, internet_has_message, np.array(counts), internet_step


# ---- Trials ----
rng = np.random.default_rng(seed=13)
n_steps = len(np.arange(0, SIMULATION_TIME * 24, DT))

delays, aligned = [], []
n_internet = 0
for trial in range(NUM_TRIALS):
    send_time, all_have_time, internet_has_message, pct, k = run_trial(rng)
    n_internet += internet_has_message
    if send_time is not None and all_have_time is not None:
        delays.append((all_have_time - send_time) / 24)

    # Pad early-stopped runs: once everyone has it, 100% of on-trail hikers do
    full = np.full(n_steps, 100.0)
    full[:len(pct)] = pct
    if send_time is not None and k is not None:
        send_step = int(round(send_time / DT))
        aligned.append((full, send_step, k))

delays = np.array(delays)
print(f"Reached internet: {n_internet} / {NUM_TRIALS} trials")
print(f"Everyone got it:  {len(delays)} / {NUM_TRIALS} trials")
if len(delays):
    print(f"Days from sending to everyone: mean {delays.mean():.2f}, "
          f"std {delays.std():.2f}, median {np.median(delays):.2f}, "
          f"min {delays.min():.2f}, max {delays.max():.2f}")

# ---- Plot 1 data: align on send time ----
s = np.array([s_ for _, s_, _ in aligned])
ks = np.array([k for _, _, k in aligned])
internet_days = (ks - s) * DT / 24
print(f"Days from sending to internet: median {np.median(internet_days):.2f}")

width_s = n_steps - s.max()  # steps every trial covers after sending
grid_s = np.stack([c[s_: s_ + width_s] for c, s_, _ in aligned])
mean_s = np.nanmedian(grid_s, axis=0)
q25_s, q75_s = np.nanpercentile(grid_s, [25, 75], axis=0)
days_s = np.arange(width_s) * DT / 24

last = np.argmax(mean_s >= 99.5) + 1 if (mean_s >= 99.5).any() else len(mean_s)
mean_s, q25_s, q75_s, days_s = (a[:last] for a in (mean_s, q25_s, q75_s, days_s))

# ---- Plot 2 data: align on internet arrival ----
offset = ks.max()
width_i = offset + n_steps - ks.min()
grid_i = np.full((len(aligned), width_i), np.nan)
inside = np.zeros((len(aligned), width_i), dtype=bool)
for i, (c, _, k) in enumerate(aligned):
    grid_i[i, offset - k: offset - k + n_steps] = c
    inside[i, offset - k: offset - k + n_steps] = True

covered = inside.all(axis=0)
g = grid_i[:, covered]
mean_i = np.nanmedian(g, axis=0)
q25_i, q75_i = np.nanpercentile(g, [25, 75], axis=0)
days_i = ((np.arange(width_i) - offset) * DT / 24)[covered]

first = np.argmax(mean_i > 0)
last = np.argmax(mean_i >= 99.5) + 1 if (mean_i >= 99.5).any() else len(mean_i)
mean_i, q25_i, q75_i, days_i = (a[first:last] for a in (mean_i, q25_i, q75_i, days_i))
pre = days_i < 0

# ---- Plot ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)

# Left: time since sent
t_net = np.median(internet_days)
net_lo, net_hi = np.percentile(internet_days, [25, 75])
ax1.axvspan(net_lo, net_hi, color="gray", alpha=0.12,
            label="Internet arrival, 25-75th pct")
ax1.fill_between(days_s, q25_s, q75_s, color="C0", alpha=0.25, lw=0,
                 label="25-75th pct across trials")
ax1.plot(days_s, mean_s, color="C0", lw=2, label="Median")
ax1.axvline(t_net, ls="--", color="gray",
            label=f"Median internet arrival ({t_net:.2f} d)")
ax1.set_xlim(0, days_s[-1])
ax1.set_xlabel("Days since message was sent")
ax1.set_ylabel("% of hikers on the trail who have the message")
ax1.set_title("Aligned on send time")
ax1.legend(loc="lower right")

# Right: relative to internet arrival
ax2.axvspan(days_i[0], 0, color="C0", alpha=0.06)
ax2.axvspan(0, days_i[-1], color="C1", alpha=0.06)
ax2.fill_between(days_i[pre], q25_i[pre], q75_i[pre], color="C0", alpha=0.25, lw=0)
ax2.fill_between(days_i[~pre], q25_i[~pre], q75_i[~pre], color="C1", alpha=0.25, lw=0)
ax2.plot(days_i[pre], mean_i[pre], color="C0", lw=2,
         label="Before internet (trail-to-trail only)")
ax2.plot(days_i[~pre], mean_i[~pre], color="C1", lw=2,
         label="After internet (entrances + trail-to-trail)")
ax2.axvline(0, ls="--", color="gray", label="Message reaches internet")
ax2.set_xlim(days_i[0], days_i[-1])
ax2.set_xlabel("Days relative to message reaching internet")
ax2.set_title("Aligned on internet arrival")
ax2.legend(loc="center left")

for ax in (ax1, ax2):
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)

fig.suptitle(f"Message reach among active hikers, {len(aligned)} trials "
             f"(bands = 25-75th percentile)")
plt.tight_layout()
plt.savefig("message_spread.png", dpi=150)
print("Saved message_spread.png")

Reached internet: 1000 / 1000 trials
Everyone got it:  1000 / 1000 trials
Days from sending to everyone: mean 2.70, std 0.61, median 2.71, min 1.21, max 4.38
Days from sending to internet: median 1.04
Saved message_spread.png


Relays